## Predicate & projection pushdown — what actually reads less data

Columnar formats (Parquet, ORC) give Spark two scan-layer wins that row formats can't:

- **Projection pushdown** — `df.select("a", "b")` reads only those columns from disk; the rest never get decoded. CSV/JSON must read whole rows even for a subset, because there's no per-column layout.
- **Predicate pushdown** — `df.filter("status = 'APPROVED'")` lets Spark skip entire **row groups** (Parquet) or **stripes** (ORC) using the min/max stats in the footer. If a group's `status` min and max are both `"DECLINED"`, it's skipped without decoding a single row.

**What pushes:** simple equality/range filters on top-level columns (`col == x`, `col > 5`, `isin`, `IS NULL`), and filters on *partition* columns — those become partition pruning, skipping whole directories (even stronger).

**What doesn't:** UDFs in the filter, expressions that compute on the column (`(col + 1) == 5` — rewrite to `col == 4`), and non-deterministic functions. Check with `.explain()` — look for `PushedFilters: [...]` in the scan node.